# Cell Profiling Anywhere

## From microscopy images to cell profiles

**Correct illumination → segment cells → measure features → save results.**

This notebook demonstrates **illumination correction, segmentation, and measurements** with `cp-anywhere`.
The main workflow uses corrected images to find cells and calculate their features.
You can use image files or NumPy arrays, work with pandas tables, and save results for later analysis.
The measurement engine runs in Rust. No CellProfiler application is needed for this example.

We use a bundled real DNA/actin image from [BBBC007](https://bbbc.broadinstitute.org/BBBC007).
No image download is needed. The sample is a small crop, not a complete biological experiment.

### Start here

From the repository root, run `uv sync --frozen --group docs`.
Open this notebook and select the repository's `.venv` Python interpreter.
Run the cells in order.

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import cp_anywhere as cp

# Work from either the repository root or this notebook's folder.
project = Path.cwd()
if not (project / "fixtures").exists():
    project = Path("../..")

sample = project / "fixtures/real/real_bbbc007"

## 1. Load two images

`load_array` reads NPY, TIFF, PNG, and other supported image formats.
These sample images are 8-bit, so dividing by `255` puts intensities between zero and one.
For your own data, change the file paths and use the correct intensity scale.

In [ ]:
dna = cp.load_array(sample / "DNA.npy").astype(float) / 255
actin = cp.load_array(sample / "Actin.npy").astype(float) / 255

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(dna, cmap="gray")
axes[0].set_title("DNA")
axes[1].imshow(actin, cmap="gray")
axes[1].set_title("Actin")
plt.show()

## 2. Correct illumination

Estimate a smooth correction function separately for DNA and actin.
Then divide each channel by its own function.
The corrected images feed the segmentation and measurement steps below.

Here we use a 64-pixel smoothing width and relative median scaling.
These are demonstration choices, not a calibrated correction for this microscope.
For real experiments, use appropriate reference images for each channel and inspect the results.

In [ ]:
dna_illumination = cp.calculate_illumination(
    dna,
    automatic_object_width="Manually",
    size_of_smoothing_filter=64,
    rescale_option="Median",
)
actin_illumination = cp.calculate_illumination(
    actin,
    automatic_object_width="Manually",
    size_of_smoothing_filter=64,
    rescale_option="Median",
)

dna_corrected = cp.apply_illumination(dna, dna_illumination)
actin_corrected = cp.apply_illumination(actin, actin_illumination)

In [ ]:
# Each pair uses the same display range so brightness stays comparable.
fig, axes = plt.subplots(2, 2, figsize=(7, 6))
axes[0, 0].imshow(dna, cmap="gray", vmin=0, vmax=1)
axes[0, 0].set_title("DNA before correction")
axes[0, 1].imshow(dna_corrected, cmap="gray", vmin=0, vmax=1)
axes[0, 1].set_title("DNA after correction")
axes[1, 0].imshow(actin, cmap="gray", vmin=0, vmax=1)
axes[1, 0].set_title("Actin before correction")
axes[1, 1].imshow(actin_corrected, cmap="gray", vmin=0, vmax=1)
axes[1, 1].set_title("Actin after correction")
plt.tight_layout()
plt.show()

## 3. Segment nuclei, cells, and cytoplasm

Find nuclei in the corrected DNA image. Grow cell regions around them using the corrected actin image.
Then remove the nuclei from those regions to get cytoplasm.

The nucleus size limits are in pixels. They suit this example, not every dataset.
These are new segmentations, not the dataset's manual ground truth.

In [ ]:
nuclei = cp.identify_primary_objects(
    dna_corrected,
    min_diameter=10,
    max_diameter=40,
    exclude_size=True,
)
cells = cp.identify_secondary_objects(actin_corrected, nuclei.labels)
cytoplasm = cp.identify_tertiary_objects(
    nuclei.labels,
    cells.labels,
    shrink_primary=False,
)

print(f"Found {nuclei.object_count} cells")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(nuclei.labels, cmap="nipy_spectral")
axes[0].set_title("Nuclei")
axes[1].imshow(cells.labels, cmap="nipy_spectral")
axes[1].set_title("Cells")
axes[2].imshow(cytoplasm.labels, cmap="nipy_spectral")
axes[2].set_title("Cytoplasm")
plt.show()

## 4. Measure cell features

Give `measure` the corrected images and the labels from segmentation.
The names become part of the output, so your channels and compartments stay identifiable.

This call measures intensity, size/shape, and neighboring cells.
All three compartments share cell IDs because we created them together.
The areas use pixel units.

In [ ]:
images = {"DNA": dna_corrected, "Actin": actin_corrected}
objects = {
    "Cells": cells.labels,
    "Nuclei": nuclei.labels,
    "Cytoplasm": cytoplasm.labels,
}

result = cp.measure(
    images,
    objects,
    families=["image_intensity", "object_intensity", "object_size_shape", "neighbors"],
)

Make a pandas table with one row per cell.
The full table contains all cell features. Here we show a few useful columns.

In [ ]:
profiles = pd.DataFrame(result.features["Cells"], index=result.object_ids["Cells"])
profiles.index.name = "cell_id"

print(f"{len(profiles)} cells, {len(profiles.columns)} features per cell")
profiles[
    [
        "AreaShape_Area",
        "Intensity_MeanIntensity_DNA",
        "Intensity_MeanIntensity_Actin",
        "Neighbors_NumberOfNeighbors_Adjacent",
    ]
].round(3)

Now use ordinary pandas plotting. Each point is one cell.

In [ ]:
profiles.plot.scatter(x="AreaShape_Area", y="Intensity_MeanIntensity_DNA")
plt.show()

## 5. Save the results

Save the cell table as CSV or Parquet. Save the labels and correction functions as NPY.
Save the corrected images as TIFF.
You can also save all compartments together, or export to SQLite.

These example files go into `docs/_build/notebook-output`.
Change `output` to a directory of your choice. Rerunning these cells replaces the example files.

In [ ]:
output = project / "docs/_build/notebook-output"
output.mkdir(parents=True, exist_ok=True)

profiles.to_csv(output / "cell_profiles.csv")
profiles.to_parquet(output / "cell_profiles.parquet")
cp.save_array(cells.labels, output / "cells.npy")
cp.save_array(dna_illumination, output / "dna_illumination.npy")
cp.save_array(actin_illumination, output / "actin_illumination.npy")
cp.save_array(
    dna_corrected, output / "dna_corrected.tiff", bit_depth="32-bit floating point"
)
cp.save_array(
    actin_corrected, output / "actin_corrected.tiff", bit_depth="32-bit floating point"
)

# Keep all compartments in one table, with one row per cell.
result.to_parquet(output / "all_compartments.parquet", mode="joined")
result.to_sqlite(output / "profiles.sqlite", if_exists="replace")

print(f"Saved files to {output}")

Read the table back whenever you need it.

In [ ]:
saved_profiles = pd.read_parquet(output / "cell_profiles.parquet")
saved_profiles.head()

## A few optional extras

The same arrays work with more of the Python API.
These independent examples do not change the profiles saved above.

### Add texture measurements

Measure actin texture within the same cells. The result has one row per cell.

In [ ]:
texture = cp.measure_texture(actin_corrected, cells.labels, image_name="Actin")
texture_profiles = pd.DataFrame(texture.objects, index=profiles.index)
texture_profiles.head()

### Already have images and labels on disk?

`measure_files` skips the segmentation steps and returns a pandas table directly.
Here we use the bundled manual-derived labels, so these measurements differ from our new segmentation.
This example keeps the original 8-bit intensity scale.

In [ ]:
file_measurements = cp.measure_files(sample / "DNA.npy", sample / "labels.npy")
file_measurements.head()

## Take it anywhere

Replace the sample files with your own aligned images.
Use existing labels or create them in Python.
Measure, inspect, and export without leaving your analysis environment.

The bundled sample is a 128 × 128 crop of BBBC007v1 Drosophila Kc167 cells, field `A9 p10`, with a CC0 waiver.
Its provenance is recorded in `reference/vendor/bbbc007/README.md`.

This notebook covers supported native Python features, not every CellProfiler module or the browser API.
Inspect segmentation before interpreting measurements.
The docs build runs every code cell, so the example stays executable.